**TUGAS 6 PRAKTIKUM BIGDATA/ ERNY KURNIAWATI / 2505060004**

**Menyalakan Spark Session**

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, sum as spark_sum, count

spark = SparkSession.builder.appName("Tugas6_ETL").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Siap untuk latihan.")

26/09/30 21:25:15 WARN Utils: Your hostname, ernykurniawati resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/30 21:25:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 21:25:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/30 21:25:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Siap untuk latihan.


In [16]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Eksplorasi Big Data") \
    .getOrCreate()

df = spark.read.csv(
    "data_transaksi_besar.csv",
    header=True,
    inferSchema=True
)

df.show(10)

+--------+-----------+----------+---------+-----------------+----------+
|order_id|customer_id|  kategori|     kota|metode_pembayaran|pendapatan|
+--------+-----------+----------+---------+-----------------+----------+
|      O0|         17|   Makanan|     Solo|              COD|    196650|
|      O1|         18| Kesehatan| Magelang|         E-Wallet|    283512|
|      O2|         14|Elektronik| Semarang|              COD|    108835|
|      O3|         13| Kesehatan|Purworejo|     Kartu Kredit|    390604|
|      O4|          2|   Fashion|Purworejo|         E-Wallet|    427224|
|      O5|          8|Elektronik| Magelang|    Transfer Bank|    390878|
|      O6|         19|   Fashion| Magelang|     Kartu Kredit|    181060|
|      O7|         14|   Fashion|     Solo|              COD|    436848|
|      O8|         17|   Fashion| Semarang|              COD|    409439|
|      O9|          1|Elektronik| Magelang|    Transfer Bank|    252580|
+--------+-----------+----------+---------+--------

**A. EXTRACT**

In [4]:
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print("Jumlah transaksi:", df_trx.count())
df_trx.printSchema()

print("Jumlah produk:", df_produk.count())
df_produk.printSchema()

print("Jumlah ulasan:", df_ulasan.count())
df_ulasan.printSchema()

Jumlah transaksi: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Jumlah produk: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Jumlah ulasan: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



**B. TRANSFORM-PENGGABUNGAN**

In [11]:
# Left join: semua transaksi tetap dipertahankan walaupun tidak punya ulasan
df_etl = df_trx.join(df_ulasan, on="order_id", how="left")

# Inner join: hanya transaksi dengan product_id yang cocok dengan data produk
df_etl = df_etl.join(df_produk, on="product_id", how="inner")

# Menghitung total pendapatan
df_etl = df_etl.withColumn(
    "total_pendapatan", col("unit_terjual") * col("harga")
)

df_etl.select(
    "order_id", "product_id", "nama_produk", "kategori",
    "unit_terjual", "harga", "rating", "total_pendapatan"
).show(10)

+--------+----------+-----------+------------+------------+------+------+----------------+
|order_id|product_id|nama_produk|    kategori|unit_terjual| harga|rating|total_pendapatan|
+--------+----------+-----------+------------+------------+------+------+----------------+
|     TX0|        21|  Produk-21|  Elektronik|           2| 25000|     1|           50000|
|     TX1|         8|   Produk-8|     Fashion|           6|250000|  NULL|         1500000|
|     TX2|        25|  Produk-25|  Elektronik|           4| 25000|     2|          100000|
|     TX3|         3|   Produk-3|     Fashion|           4| 75000|     5|          300000|
|     TX4|        19|  Produk-19|Rumah Tangga|           6| 75000|  NULL|          450000|
|     TX5|        10|  Produk-10|  Elektronik|           3| 75000|     5|          225000|
|     TX6|        26|  Produk-26|   Kesehatan|           6| 75000|     4|          450000|
|     TX7|         4|   Produk-4|     Makanan|           1|150000|  NULL|          150000|

**C. TRANFORM - PENANGANAN DATA KOSONG & PENGAYAAN**

In [7]:
# Menentukan ada atau tidaknya ulasan sebelum null diisi
df_etl = df_etl.withColumn(
    "ada_ulasan", col("rating").isNotNull()
)

# Mengisi rating yang kosong dengan 0
df_etl = df_etl.na.fill({"rating": 0})

df_etl.select(
    "order_id", "nama_produk", "kategori", "rating",
    "ada_ulasan", "total_pendapatan"
).show(10)

+--------+-----------+------------+------+----------+----------------+
|order_id|nama_produk|    kategori|rating|ada_ulasan|total_pendapatan|
+--------+-----------+------------+------+----------+----------------+
|     TX0|  Produk-21|  Elektronik|     1|      true|           50000|
|     TX1|   Produk-8|     Fashion|     0|      true|         1500000|
|     TX2|  Produk-25|  Elektronik|     2|      true|          100000|
|     TX3|   Produk-3|     Fashion|     5|      true|          300000|
|     TX4|  Produk-19|Rumah Tangga|     0|      true|          450000|
|     TX5|  Produk-10|  Elektronik|     5|      true|          225000|
|     TX6|  Produk-26|   Kesehatan|     4|      true|          450000|
|     TX7|   Produk-4|     Makanan|     0|      true|          150000|
|     TX8|   Produk-7|  Elektronik|     5|      true|          150000|
|     TX9|  Produk-30|  Elektronik|     0|      true|          250000|
+--------+-----------+------------+------+----------+----------------+
only s

**D. LOAD**

In [8]:
hdfs_path = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"

df_etl.write.mode("overwrite").partitionBy("kategori").parquet(hdfs_path)
print("Hasil ETL berhasil disimpan ke HDFS.")

# Verifikasi folder hasil
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

Hasil ETL berhasil disimpan ke HDFS.
-rw-r--r--   3 ernykurniawati supergroup          0 2026-09-30 21:45 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - ernykurniawati supergroup          0 2026-09-30 21:45 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 ernykurniawati supergroup      16170 2026-09-30 21:45 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-a943032c-6ffe-4fe8-ba0a-bc3c4cefde74.c000.snappy.parquet
drwxr-xr-x   - ernykurniawati supergroup          0 2026-09-30 21:45 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 ernykurniawati supergroup      11008 2026-09-30 21:45 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-a943032c-6ffe-4fe8-ba0a-bc3c4cefde74.c000.snappy.parquet
drwxr-xr-x   - ernykurniawati supergroup          0 2026-09-30 21:45 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 ernykurniawati supergroup      10271 2026-09-30 21:45 /user/mahasiswa/tugas6/hasil_etl/kategori=Ke

In [9]:
# Membaca kembali hasil dari HDFS dan mengecek jumlah baris
df_hasil = spark.read.parquet(hdfs_path)
print("Jumlah baris hasil ETL:", df_hasil.count())
df_hasil.show(5)

Jumlah baris hasil ETL: 5000
+----------+--------+------------+-------------------+------+------+-----------+----------------+----------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|nama_produk|total_pendapatan|ada_ulasan|  kategori|
+----------+--------+------------+-------------------+------+------+-----------+----------------+----------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Produk-21|           50000|      true|Elektronik|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Produk-25|          100000|      true|Elektronik|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Produk-10|          225000|      true|Elektronik|
|         7|     TX8|           2|2026-10-09 00:00:00|     5| 75000|   Produk-7|          150000|      true|Elektronik|
|        30|     TX9|           1|2026-10-27 00:00:00|     0|250000|  Produk-30|          250000|      true|Elektronik|
+----------

**E. INSIGHT AKHIR**

In [10]:
insight = df_hasil.groupBy("kategori").agg(
    count("order_id").alias("jumlah_transaksi"),
    spark_sum(when(col("ada_ulasan") == True, 1).otherwise(0)).alias("jumlah_dengan_ulasan")
).withColumn(
    "persentase_ulasan",
    col("jumlah_dengan_ulasan") / col("jumlah_transaksi") * 100
).orderBy("persentase_ulasan")

insight.show()
print("Kategori dengan persentase transaksi berulasan paling rendah:")
insight.select("kategori", "persentase_ulasan").show(1, truncate=False)

+------------+----------------+--------------------+-----------------+
|    kategori|jumlah_transaksi|jumlah_dengan_ulasan|persentase_ulasan|
+------------+----------------+--------------------+-----------------+
|  Elektronik|            1653|                1653|            100.0|
|     Fashion|            1035|                1035|            100.0|
|Rumah Tangga|             815|                 815|            100.0|
|   Kesehatan|             961|                 961|            100.0|
|     Makanan|             536|                 536|            100.0|
+------------+----------------+--------------------+-----------------+

Kategori dengan persentase transaksi berulasan paling rendah:
+----------+-----------------+
|kategori  |persentase_ulasan|
+----------+-----------------+
|Elektronik|100.0            |
+----------+-----------------+
only showing top 1 row



**EXPLORASI**

In [17]:
df.groupBy("kategori").count().show()

[Stage 43:>                                                         (0 + 1) / 1]

+------------+-----+
|    kategori|count|
+------------+-----+
|  Elektronik| 3971|
|     Fashion| 4050|
|     Makanan| 4000|
|Rumah Tangga| 3940|
|   Kesehatan| 4039|
+------------+-----+



In [18]:
#Meenghitung rata-rata pendapatan
from pyspark.sql.functions import avg
df.select(avg("pendapatan").alias("rata_rata_pendapatan")).show()

+--------------------+
|rata_rata_pendapatan|
+--------------------+
|        274103.43925|
+--------------------+



In [19]:
#Menghitung jumlah transaksi berdasarkan metode pembayaran
df.groupBy("metode_pembayaran").count().show()

+-----------------+-----+
|metode_pembayaran|count|
+-----------------+-----+
|              COD| 5064|
|    Transfer Bank| 5044|
|     Kartu Kredit| 4948|
|         E-Wallet| 4944|
+-----------------+-----+



In [20]:
#Menghitung Kategori dengan Pendapatan Terbesar
df.groupBy("kategori") \
  .agg(sum("pendapatan").alias("total_pendapatan")) \
  .orderBy("total_pendapatan", ascending=False) \
  .show()

+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|   Kesehatan|      1121008753|
|     Makanan|      1102209801|
|     Fashion|      1097850443|
|  Elektronik|      1085313525|
|Rumah Tangga|      1075686263|
+------------+----------------+

